In [14]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/kaggle_data/optiver-realized-volatility-prediction'

Mounted at /content/drive


# Data Breakdown

### Book Data
* `time_id`: The dataset is composed of 10 minute blocks. Each 10 minute block is uniquely identified by its `time_id`. 
* `seconds_in_bucket`: The amount of seconds that has elapsed since the beginning of the bucket identified by `time_id`. This is a snapshot of the market at this particular time. If a particular second is missing, it is assumed that the market is unchanged from the last entry. 
* `[bid/ask]_price[1/2]`: Exactly what it sounds like. `bid_price2` < `bid_price1` < `ask_price1` < `ask_price2`. 
* `stock_id`: Unique stock identifier. 

### Trade Data
* `time_id`: The same as in Book Data
* `seconds_in_bucket`: Similar to `seconds_in_bucket` from Book Data. This is the second for which the trade data is considered. I.e. all other data summarizes all the trade information over the time period `[seconds_in_bucket, seconds_in_bucket + 1)`. If a particular second is missing, it is not assumed that the market is unchanged from the last entry. Each entry signifies market activity. If there is no entry for a particular second, no trades were executed over that second.  
* `price`: This is the volume weighted price over the second. 
* `size`: This is the number of shares traded. 
* `order_count`: This is the number of unique trade orders placed in that second. 

# Encoding

This encoding is inspired by [this paper](https://arxiv.org/pdf/2304.02472). It differs, since we are not (yet) interested in applying a pre-trained image model, which is what they are trying to do in the end. 

In this encoding, our x axis is time, and y axis is price. We discretize the price axis, and aggregate the data correspondingly. Our channels are as follow:
* `bid_size1`
* `bid_size2`
* `ask_size1`
* `ask_size2`
* `size` 
* `order_count`

We will have to have a different encoding scheme to add more features. 
The shape of one sample is [600, 600, 6]. 

# BAD ASSUMPTION THAT WE WILL USE: (`stock_id`, `time_id`) samples are i.i.d. 
We can think about fixing it later

In [ ]:
class OrderFlowDataset(Dataset):
    def __init__(self, target_csv_path, book_path, trade_path, transform=None):
        self.target = pd.read_csv(target_csv_path)
        self.index_map = self.target[['stock_id', 'time_id']].to_numpy()
        self.target = self.target.to_numpy()

        self.book_path = book_path
        self.trade_path = trade_path
        self.transform = transform

    def __len__(self):
        return len(self.index_map)
    
    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        stock_id, time_id = self.index_map[idx]
        book_data = pd.read_parquet(f"{self.book_path}/stock_id={stock_id}", filters=[('time_id', '==', time_id)])
        trade_data = pd.read_parquet(f"{self.trade_path}/stock_id={stock_id}", filters=[('time_id', '==', time_id)])
        target = self.target[idx, 2]

        sample = {"book": book_data, "trade": trade_data, "r_vol": target}

        if self.transform:
            sample = self.transform(sample)
        
        return sample

Here is an example of how to use the dataset:

In [83]:
of_datset = OrderFlowDataset(f"{DATA_DIR}/train.csv", f"{DATA_DIR}/book_train.parquet", f"{DATA_DIR}/trade_train.parquet")
sample = of_datset[0]
print(f"Book Sample: \n{sample["book"]}\nTrade Sample: \n {sample["trade"]}\nRealized Volatility Sample Target: \n{sample["r_vol"]}")

Book Sample: 
     time_id  seconds_in_bucket  bid_price1  ask_price1  bid_price2  \
0          5                  0    1.001422    1.002301    1.001370   
1          5                  1    1.001422    1.002301    1.001370   
2          5                  5    1.001422    1.002301    1.001370   
3          5                  6    1.001422    1.002301    1.001370   
4          5                  7    1.001422    1.002301    1.001370   
..       ...                ...         ...         ...         ...   
297        5                585    1.003129    1.003749    1.003025   
298        5                586    1.003129    1.003749    1.002612   
299        5                587    1.003129    1.003749    1.003025   
300        5                588    1.003129    1.003749    1.002612   
301        5                593    1.003129    1.003749    1.003025   

     ask_price2  bid_size1  ask_size1  bid_size2  ask_size2  
0      1.002353          3        226          2        100  
1      1.

# Need to Implement: 
* Transform function to map book sample and trade sample to multichannel image. 
* Dataloader, and also understand the test train split formulation in pytorch 
* (Later) Rest of Feature engineering, likely added as part of the transform function pipeline